# Fintech Privacy Filter — Training Notebook

Fine-tunes `OpenMed/privacy-filter-nemotron` on the `msdakot/fintech-privacy-pii` dataset.

**Runtime:** ~2–4 hours on T4  
**Before running:** Set `HF_TOKEN` in Colab Secrets (key icon in left sidebar)

**If Colab disconnects mid-run:** Re-run from Cell 5 (Mount Drive) — training resumes from the latest Drive checkpoint automatically because `--output-dir` points to Drive.

---
### Checklist before running
- [ ] Runtime → Change runtime type → T4 GPU
- [ ] Left sidebar → Key icon → Add secret `HF_TOKEN` with your write-access token
- [ ] Run cells top to bottom

## Cell 1 — Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — switch runtime to T4 before continuing.')

## Cell 2 — Mount Google Drive (checkpoint persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/fintech-pii-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

## Cell 3 — Install dependencies

In [ ]:
# Install opf and datasets; torch is pre-installed on Colab T4
!pip install -q 'opf @ git+https://github.com/openai/privacy-filter.git' datasets huggingface_hub
print('Dependencies installed.')

## Cell 4 — Authenticate with HuggingFace

In [ ]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import HfApi
user = HfApi(token=HF_TOKEN).whoami()
print(f"Authenticated as: {user['name']}")

## Cell 5 — Download base model checkpoint (OpenMed/privacy-filter-nemotron)

In [ ]:
from huggingface_hub import snapshot_download

BASE_MODEL_DIR = '/content/privacy-filter-nemotron'

if not os.path.exists(BASE_MODEL_DIR):
    print('Downloading OpenMed/privacy-filter-nemotron...')
    snapshot_download(
        repo_id='OpenMed/privacy-filter-nemotron',
        local_dir=BASE_MODEL_DIR,
        token=HF_TOKEN,
    )
    print('Download complete.')
else:
    print('Base model already cached.')

import os
print('Checkpoint files:', os.listdir(BASE_MODEL_DIR))

## Cell 6 — Load dataset from HF Hub and export to opf JSONL

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

def record_to_opf(record):
    """Convert HF dataset record → opf JSONL format.
    
    HF format:  spans = [{start, end, label}, ...]
    opf format: spans = {label: [[start, end], ...], ...}
    """
    spans_dict = {}
    for span in record.get('spans', []):
        label = span['label']
        spans_dict.setdefault(label, []).append([span['start'], span['end']])
    return {'text': record['text'], 'spans': spans_dict}

print('Loading msdakot/fintech-privacy-pii from HF Hub...')
ds = load_dataset('msdakot/fintech-privacy-pii', token=HF_TOKEN)

for split in ('train', 'validation', 'test'):
    path = DATA_DIR / f'{split}.jsonl'
    with open(path, 'w', encoding='utf-8') as f:
        for record in ds[split]:
            f.write(json.dumps(record_to_opf(record), ensure_ascii=False) + '\n')
    print(f'  {split}: {len(ds[split])} records → {path}')

print('Export complete.')

## Cell 7 — Write label space JSON (65 entity types)

In [ ]:
import json

# 55 labels inherited from OpenMed/privacy-filter-nemotron + 10 new fintech labels
label_space = {
    "span_class_names": [
        "O",
        "account_number", "age", "api_key", "bank_routing_number",
        "biometric_identifier", "blood_type", "certificate_license_number",
        "city", "company_name", "coordinate", "country", "county",
        "credit_debit_card", "customer_id", "cvv", "date", "date_of_birth",
        "date_time", "device_identifier", "education_level", "email",
        "employee_id", "employment_status", "fax_number", "first_name",
        "gender", "health_plan_beneficiary_number", "http_cookie",
        "ipv4", "ipv6", "language", "last_name", "license_plate",
        "mac_address", "medical_record_number", "national_id", "occupation",
        "password", "phone_number", "pin", "political_view", "postcode",
        "race_ethnicity", "religious_belief", "sexuality", "ssn", "state",
        "street_address", "swift_bic", "tax_id", "time", "unique_id",
        "url", "user_name", "vehicle_identifier",
        # New fintech labels
        "iban", "bban", "lei", "isin", "cusip",
        "swift_mt_ref", "policy_number", "vat_number", "loan_number", "crypto_address"
    ]
}

LABEL_SPACE_PATH = '/content/label_space.json'
with open(LABEL_SPACE_PATH, 'w') as f:
    json.dump(label_space, f, indent=2)

print(f'Label space written: {len(label_space["span_class_names"]) - 1} entity types + O')

## Cell 8 — Run training

Expected runtime: **2–4 hours** on T4.  
Checkpoints save to Google Drive — safe to disconnect and reconnect.

In [ ]:
import subprocess, sys

cmd = [
    'opf', 'train', str(DATA_DIR / 'train.jsonl'),
    '--validation-dataset', str(DATA_DIR / 'validation.jsonl'),
    '--checkpoint', BASE_MODEL_DIR,
    '--label-space-json', LABEL_SPACE_PATH,
    '--output-dir', CHECKPOINT_DIR,
    '--overwrite-output',
    '--epochs', '3',
    '--batch-size', '2',
    '--grad-accum-steps', '4',
    '--learning-rate', '1e-4',
    '--weight-decay', '0.0',
    '--output-param-dtype', 'bf16',
]

print('Command:', ' '.join(cmd))
print('Training started — output below...')
print('-' * 60)

result = subprocess.run(cmd)

if result.returncode == 0:
    print('\nTraining complete.')
else:
    print(f'\nTraining exited with code {result.returncode}')
    sys.exit(result.returncode)

## Cell 9 — Verify checkpoint

In [ ]:
import os, json

print('Checkpoint contents:', os.listdir(CHECKPOINT_DIR))

summary_path = os.path.join(CHECKPOINT_DIR, 'finetune_summary.json')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print('\nTraining summary:')
    print(json.dumps(summary, indent=2))
else:
    print('No finetune_summary.json found — training may not have completed.')

## Cell 10 — Push trained model to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi, create_repo

HF_MODEL_REPO = 'msdakot/fintech-privacy-filter-v0'
api = HfApi(token=HF_TOKEN)

create_repo(HF_MODEL_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)
print(f'Repo ready: {HF_MODEL_REPO}')

api.upload_folder(
    folder_path=CHECKPOINT_DIR,
    repo_id=HF_MODEL_REPO,
    repo_type='model',
    commit_message='Initial fine-tuned checkpoint — fintech-privacy-filter-v0',
)

print(f'\nModel pushed to: https://huggingface.co/{HF_MODEL_REPO}')

## Cell 11 — Quick sanity check (inference on 3 examples)

In [ ]:
from huggingface_hub import snapshot_download

# Pull the just-pushed model back to verify it's loadable
local_model = snapshot_download(repo_id=HF_MODEL_REPO, token=HF_TOKEN)
print('Model downloaded to:', local_model)

import subprocess

test_texts = [
    'Wire transfer to IBAN DE89370400440532013000 completed.',
    'The LEI 529900T8BM49AURSDO55 was submitted to the trade repository.',
    'Contact john.smith@acmecorp.com for invoice queries.',
]

for text in test_texts:
    result = subprocess.run(
        ['opf', 'redact', '--checkpoint', local_model, text],
        capture_output=True, text=True
    )
    print(f'Input:  {text}')
    print(f'Output: {result.stdout.strip()}')
    print()